## Prerequisites

In [ ]:
# Check CUDA version
nvcc --version

# Check GPU
nvidia-smi

# Python version (3.10+ recommended)
python --version

## Install TensorRT

* Then upgrade your CUDA toolkit to 12.x (your driver already supports 12.6, so this is safe)

In [ ]:
# Remove old CUDA toolkit
sudo apt remove --purge cuda-toolkit-11-5
sudo apt autoremove

# Install CUDA 12.6 toolkit
wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
sudo dpkg -i cuda-keyring_1.1-1_all.deb
sudo apt update
sudo apt install cuda-toolkit-12-6

# Add to PATH
echo 'export PATH=/usr/local/cuda-12.6/bin:$PATH' >> ~/.bashrc
echo 'export LD_LIBRARY_PATH=/usr/local/cuda-12.6/lib64:$LD_LIBRARY_PATH' >> ~/.bashrc
source ~/.bashrc

*  Install cuDNN 9.x (required by TRT 10)

In [ ]:
!sudo apt install libcudnn9-cuda-12 libcudnn9-dev-cuda-12

In [ ]:
# Recommended: pip install (includes Python bindings + core libs)
!pip install tensorrt tensorrt-lean tensorrt-dispatch

# Also install pycuda for buffer management
!pip install pycuda

# Install torch-tensorrt if you want PyTorch integration
!pip install torch-tensorrt

In [2]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 4.9 MB/s eta 0:00:005.3 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 4.6 MB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 4.7 MB/s eta 0:00:00m eta 0:00:010:00:01
  Using cached torch-2.11.0-cp310-cp310-manylinux_2_28_x86_64.whl (530.6 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 KB 5.3 MB/s eta 0:00:005.8 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 824.0/824.0 KB 6.1 MB/s eta 0:00:006.9 MB/s eta 0:00:01
  Using cached pillow-12.1.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (7.0 MB)
  Using cached torchvision-0.26.0-cp310-cp310-manylinux_2_28_x86_64.whl (7.5 MB)
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 5.8 MB/s eta 0:00:00m eta 0:00:010:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0

In [ ]:
# Uninstall the wrong PyTorch first
!pip uninstall torch torchvision torchaudio -y

# Install PyTorch cu124 (works on your CUDA 12.6 driver)
!pip install torch torchvision torchaudio \
  --index-url https://download.pytorch.org/whl/cu124

In [7]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("PyTorch built with CUDA:", torch.version.cuda)

PyTorch version: 2.6.0+cu124
CUDA available: True
PyTorch built with CUDA: 12.4


In [1]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
print('model loaded')

model loaded


In [5]:
results = model('data/img1.png', half = True)


image 1/1 /home/rakshith/Tutorial/tensorrt/data/img1.png: 384x640 2 boats, 41.1ms
Speed: 1.2ms preprocess, 41.1ms inference, 6.8ms postprocess per image at shape (1, 3, 384, 640)


In [6]:
model.export(format = 'engine', half = True)

WARNING ⚠️ TensorRT requires GPU export, automatically assigning device=0
Ultralytics 8.4.31 🚀 Python-3.10.12 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7836MiB)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.3/237.3 KB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 MB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 KB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━

PosixPath('yolov8n.engine')

In [8]:
trt_model = YOLO("yolov8n.engine")

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.


In [9]:

results = trt_model("https://ultralytics.com/images/bus.jpg", show=True)

Loading yolov8n.engine for TensorRT inference...
[03/29/2026-21:06:09] [TRT] [I] Loaded engine size: 8 MiB
[03/29/2026-21:06:09] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +9, now: CPU 0, GPU 15 (MiB)

Found https://ultralytics.com/images/bus.jpg locally at bus.jpg


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/rakshith/Tutorial/tensorrt/tensorrt/lib/python3.10/site-packages/cv2/qt/plugins"
QFontDatabase: Cannot find font directory /home/rakshith/Tutorial/tensorrt/tensorrt/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/rakshith/Tutorial/tensorrt/tensorrt/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/rakshith/Tutorial/tensorrt/tensorrt/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/rakshith/Tutorial/tensorrt/tensorrt/lib/pytho

image 1/1 /home/rakshith/Tutorial/tensorrt/bus.jpg: 640x640 4 persons, 1 bus, 1.2ms
Speed: 1.8ms preprocess, 1.2ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 640)
